# Domain 3: Integration
**Exam**: Claude Certified Architect – Professional (CCAR-P)
**Weighting**: 19% of exam
**Depth**: deep

## Domain Overview

This domain covers evaluating tool/agent configuration for capability bloat, analyzing
authentication/authorization gaps, evaluating accuracy-latency tradeoffs, selecting monitoring
strategies at scale, designing RAG pipelines, matching retrieval strategy to data shape, choosing
connection protocols (MCP, API/CLI, agent-to-agent), and evaluating progressive discovery vs.
monolithic context strategy.

**Why it matters**: at 19% this is the largest domain — professional architects must integrate
Claude into real enterprise systems securely, observably, and at the right level of retrieval
sophistication.

**Key concepts to know**: least-privilege tool configuration, auth/authz gap analysis, RAG pipeline
design (chunking/indexing/retrieval), integration protocol selection, and progressive discovery vs.
monolithic context.

In [ ]:
# Setup: a small helper so every cell in this notebook runs end-to-end,
# whether or not you have an ANTHROPIC_API_KEY configured.
import os

def call_claude(prompt, system=None, model="claude-sonnet-4-5", max_tokens=600):
    """Calls the real Claude API if ANTHROPIC_API_KEY is set; otherwise returns a
    simulated response so this notebook is runnable without API access."""
    try:
        import anthropic
        api_key = os.environ.get("ANTHROPIC_API_KEY")
        if not api_key:
            raise RuntimeError("no ANTHROPIC_API_KEY set")
        client = anthropic.Anthropic(api_key=api_key)
        kwargs = {"model": model, "max_tokens": max_tokens, "messages": [{"role": "user", "content": prompt}]}
        if system:
            kwargs["system"] = system
        resp = client.messages.create(**kwargs)
        return resp.content[0].text
    except Exception as e:
        return (f"[SIMULATED RESPONSE - set ANTHROPIC_API_KEY and `pip install anthropic` "
                f"to call the real API]\nPrompt received ({len(prompt)} chars): {prompt[:150]}...")

print("call_claude() ready. Set ANTHROPIC_API_KEY as an environment variable to hit the real API.")

## Key Concepts

### Evaluate tool/agent configuration for capability bloat and authentication gaps

Capability bloat (too many tools, overly broad permissions) increases both unreliability and
security risk. An authorization audit should check whether each tool's granted scope matches the
minimum required for the agent's role, and whether authentication is enforced per-tool or only at
the session level (a common gap).

In [ ]:
def audit_tool_permissions(tool_name: str, granted_scopes: list[str], minimum_required: list[str]) -> dict:
    excess = [s for s in granted_scopes if s not in minimum_required]
    return {"tool": tool_name, "excess_scopes": excess, "over_provisioned": len(excess) > 0}

print(audit_tool_permissions("crm_agent_tool", ["read:contacts", "write:contacts", "delete:contacts"], ["read:contacts"]))

### Evaluate accuracy-latency trade-offs and observability at scale

Higher-accuracy configurations (larger models, more retrieval steps, verification passes)
typically cost more latency. At scale, observability must move beyond ad-hoc logging to structured
tracing (per-request trace IDs, token/cost/latency metrics, sampled human review) so issues can be
diagnosed across millions of requests.

In [ ]:
def accuracy_latency_tradeoff(verification_passes: int) -> dict:
    base_latency_ms = 800
    return {
        "passes": verification_passes,
        "estimated_latency_ms": base_latency_ms * (1 + 0.6 * verification_passes),
        "estimated_accuracy_gain_pct": min(15, verification_passes * 6),
    }

for passes in [0, 1, 2]:
    print(accuracy_latency_tradeoff(passes))

### Design a RAG pipeline with appropriate chunking, indexing, and retrieval strategies

Chunk size and overlap should match document structure (e.g., smaller chunks for dense
legal clauses, larger for narrative text). Indexing strategy (semantic/vector, keyword/BM25, or
hybrid) should match query patterns: keyword-heavy lookups favor BM25/hybrid; conceptual questions
favor semantic search.

In [ ]:
def choose_retrieval_strategy(query_style: str, data_shape: str) -> str:
    if query_style == "exact term/code lookup":
        return "keyword (BM25) or hybrid"
    if data_shape == "dense structured clauses":
        return "semantic search with small, precise chunks"
    return "semantic search with moderate chunk size + overlap"

print(choose_retrieval_strategy("exact term/code lookup", "logs"))
print(choose_retrieval_strategy("conceptual question", "dense structured clauses"))

### Evaluate connection protocols and select the appropriate integration mechanism

MCP suits reusable, standardized tool/resource access shared across multiple Claude
applications. A direct API/CLI integration suits a single, tightly-coupled application. Agent-to-agent
protocols suit cross-organization or cross-platform agent collaboration where neither side controls
the other's implementation.

In [ ]:
def choose_integration_mechanism(shared_across_apps: bool, cross_org: bool) -> str:
    if cross_org:
        return "agent-to-agent protocol"
    if shared_across_apps:
        return "MCP server"
    return "direct API/CLI integration"

print(choose_integration_mechanism(shared_across_apps=True, cross_org=False))
print(choose_integration_mechanism(shared_across_apps=False, cross_org=True))

### Evaluate progressive discovery vs. monolithic context strategy

Monolithic context (loading everything up front) is simple but wastes tokens and risks
"lost in the middle" effects at scale. Progressive discovery (exposing a catalog via MCP resources,
then fetching detail on demand) scales better for large knowledge bases but adds round-trip latency
per lookup — the right choice depends on knowledge base size and query diversity.

In [ ]:
def choose_context_strategy(knowledge_base_size_docs: int, query_diversity: str) -> str:
    if knowledge_base_size_docs > 1000 or query_diversity == "high":
        return "progressive discovery via MCP resources / on-demand retrieval"
    return "monolithic context - load relevant docs up front"

print(choose_context_strategy(50, "low"))
print(choose_context_strategy(50_000, "high"))

## Mini Hands-On Project

**Covers**: all Domain 3 sub-objectives.

**Scenario**: Integrate Claude with a 200,000-document legal knowledge base, shared across 3
internal applications, with strict access controls.

### Step 1: Audit and scope tool permissions

In [ ]:
print(audit_tool_permissions("legal_doc_search_tool", ["read:cases", "write:cases"], ["read:cases"]))

### Step 2: Design the RAG and context strategy

In [ ]:
print(choose_retrieval_strategy("conceptual question", "dense structured clauses"))
print(choose_context_strategy(200_000, "high"))

### Step 3: Choose the integration mechanism and estimate the latency/accuracy tradeoff

In [ ]:
print(choose_integration_mechanism(shared_across_apps=True, cross_org=False))
print(accuracy_latency_tradeoff(1))

## Lab Exercises

### Exercise 1: Permission audit for 3 tools

**Objective**: Audit 3 tools for capability bloat and propose scoped-down permissions.

**Produce**: 3 audits with recommended minimal scopes.

In [ ]:
tools = [
    ("email_tool", ["read:inbox", "send:email", "delete:inbox"], ["send:email"]),
    ("billing_tool", ["read:invoices", "write:invoices", "refund:payments"], ["read:invoices"]),
    ("search_tool", ["read:documents"], ["read:documents"]),
]
# TODO: call audit_tool_permissions for each and print the result
for name, granted, minimum in tools:
    pass

# Expected output: email_tool and billing_tool flagged as over-provisioned; search_tool correctly not flagged.

In [ ]:
# STRETCH GOAL: write a remediation plan (which scopes to remove) for each over-provisioned tool.
remediation_plans = {}  # TODO
print(remediation_plans)

# Expected output: A specific list of scopes to remove per over-provisioned tool.

### Exercise 2: Latency budget planning

**Objective**: Given a 2-second latency budget, determine the max verification passes allowed.

**Produce**: A max-passes calculation with reasoning.

In [ ]:
latency_budget_ms = 2000
# TODO: use accuracy_latency_tradeoff to find the maximum verification passes that stay under budget
max_passes = None
for p in range(5):
    pass  # TODO: check accuracy_latency_tradeoff(p)["estimated_latency_ms"] against latency_budget_ms
print(max_passes)

# Expected output: Correctly identifies the largest pass count whose estimated latency stays under 2000ms.

In [ ]:
# STRETCH GOAL: given the accuracy gain tapers off (diminishing returns), argue whether it's worth
# using the maximum allowed passes or stopping earlier.
recommendation = "..."  # TODO

# Expected output: A reasoned recommendation, likely favoring fewer passes given diminishing accuracy returns.

### Exercise 3: RAG strategy for 4 data types

**Objective**: Choose a retrieval strategy for 4 different data/query combinations.

**Produce**: 4 strategy choices with reasoning.

In [ ]:
cases = [
    ("Searching error codes in log files", "exact term/code lookup", "logs"),
    ("Answering conceptual policy questions", "conceptual question", "narrative policy text"),
    ("Finding a specific clause by number", "exact term/code lookup", "dense structured clauses"),
    ("Summarizing themes across many reports", "conceptual question", "narrative reports"),
]
# TODO: call choose_retrieval_strategy for each
for desc, style, shape in cases:
    pass

# Expected output: Sensible strategy per case, e.g. keyword/hybrid for log error codes, semantic for conceptual summarization.

In [ ]:
# STRETCH GOAL: propose a chunk size (in tokens, rough estimate) for the "dense structured clauses" case
# vs. the "narrative reports" case, and justify the difference.
chunk_sizes = {}  # TODO
print(chunk_sizes)

# Expected output: A smaller chunk size for dense clauses and a larger one for narrative reports, with justification.

### Exercise 4: Integration mechanism selection

**Objective**: Choose the integration mechanism for 3 new scenarios.

**Produce**: 3 mechanism choices with reasoning.

In [ ]:
scenarios = [
    ("A single internal app calling Claude directly", False, False),
    ("A tool shared across 5 internal Claude apps", True, False),
    ("Collaborating with a partner company's independent agent system", False, True),
]
# TODO: call choose_integration_mechanism for each
for desc, shared, cross_org in scenarios:
    pass

# Expected output: Direct API/CLI, MCP, and agent-to-agent chosen correctly for the 3 respective scenarios.

In [ ]:
# STRETCH GOAL: identify one security consideration unique to the agent-to-agent case that
# wouldn't apply to the internal MCP case.
unique_consideration = "..."  # TODO

# Expected output: A consideration such as verifying the identity/trust level of an external, non-controlled agent.

### Exercise 5: Progressive discovery vs. monolithic decision at scale

**Objective**: Decide context strategy for a growing knowledge base over time (3 size stages).

**Produce**: 3 decisions showing when the strategy should change.

In [ ]:
growth_stages = [
    ("Launch", 200, "low"),
    ("6 months in", 5_000, "medium"),
    ("2 years in", 80_000, "high"),
]
# TODO: call choose_context_strategy for each stage
for stage, size, diversity in growth_stages:
    pass

# Expected output: Monolithic context recommended at launch, progressive discovery recommended by the later, larger stages.

In [ ]:
# STRETCH GOAL: identify the approximate size/diversity threshold where you'd plan the migration
# from monolithic to progressive discovery, and what would break if you waited too long.
migration_trigger = "..."  # TODO
failure_if_delayed = "..."  # TODO

# Expected output: A concrete threshold and a specific failure mode (e.g., token cost explosion, lost-in-the-middle errors) if migration is delayed.

## Self-Assessment: Integration

Answer each question, then check the Answers cell below.


**Q1.** What is 'capability bloat' and why is it both a reliability and a security risk?

**Q2.** Why is structured tracing/observability more important at scale than ad-hoc logging?

**Q3.** What retrieval strategy is best suited to exact term/code lookups versus conceptual questions?

**Q4.** When should MCP be preferred over a direct API/CLI integration?

**Q5.** What is the tradeoff between monolithic context and progressive discovery as a knowledge base grows?

## Answers


**A1.** Granting more tools/permissions than an agent's role requires; it increases both misuse/misselection risk and the attack surface if the agent is compromised.

**A2.** Because at high request volume, you need structured, queryable data (trace IDs, metrics, sampling) to diagnose issues across millions of requests, not manual log review.

**A3.** Keyword (BM25) or hybrid search for exact lookups; semantic (vector) search for conceptual questions.

**A4.** When the integration needs to be reused/shared across multiple Claude applications as a standardized tool/resource interface.

**A5.** Monolithic context is simpler but wastes tokens and risks 'lost in the middle' effects as the knowledge base grows; progressive discovery scales better but adds per-lookup latency.

---
**Next**: [Domain 4: Evaluation, Testing & Optimization](./04-evaluation-testing-and-optimization.ipynb)
**Back**: [README](./README.md)